# Exercises — XGBoost and model evaluation

Run the setup and shared split below. The data consists of independent hourly scenarios, not a chronological time series. The TODO cells are safe to run before you fill them in: they are intentionally empty and do not raise exceptions. Suggested answers are in `solutions.ipynb`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

SEED = 42
FEATURES = ["temperature_c", "humidity", "wind_kmh", "hour", "weekday", "season"]
TARGET = "rentals"

def find_data_file(filename="bike_demand.csv"):
    # In Jupyter, cwd is commonly the repository root or this notebook's folder.
    for start in (Path.cwd(), *Path.cwd().parents):
        candidate = start / "data" / filename
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find data/" + filename + " in the current directory or its parents. "
        "Open Jupyter from the repository or place the supplied CSV in data/."
    )

data_path = find_data_file()
df = pd.read_csv(data_path)
missing = sorted(set(FEATURES + [TARGET]) - set(df.columns))
if missing:
    raise ValueError(f"{data_path} is missing required columns: {missing}")
df = df[FEATURES + [TARGET]].dropna().copy()
print(f"Loaded {len(df):,} complete scenario rows from {data_path}")
df.head()


In [ ]:
# Treat hour and weekday as labels, not measurements with meaningful linear distance.
# One-hot columns are shared across all models so the comparison uses the same inputs.
categorical = ["hour", "weekday", "season"]
numeric = ["temperature_c", "humidity", "wind_kmh"]
encoder = ColumnTransformer(
    [("numeric", "passthrough", numeric),
     ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical)],
    verbose_feature_names_out=False,
)
X = df[FEATURES]
y = df[TARGET].astype(float)

# First hold out the final test set. Then split the remaining development rows
# into model-fit and early-stopping validation subsets.
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.1875, random_state=SEED
)  # 0.1875 * 0.80 = 0.15 of all rows
print(f"train={len(X_train)}, validation={len(X_val)}, test={len(X_test)}")


## Exercise 1 — Derive a leaf update

For \(q(w)=Gw+\frac12(H+\lambda)w^2+\alpha|w|\), derive the minimizer. Explain why the answer becomes zero when \(|G|\le\alpha\), and state what happens as \(\lambda\) increases.

In [ ]:
# TODO: Write a short derivation in a Markdown cell, or use this space for notes.


## Exercise 2 — Recover a split gain

For L2-only regularization, compute the approximate gain for a split with \((G_L,H_L)=(-8,10)\), \((G_R,H_R)=(5,8)\), \((G_P,H_P)=(-3,18)\), \(\lambda=2\), and \(\gamma=0.4\). Is the split preferred by this approximate objective? What does that answer fail to tell you?

In [ ]:
# TODO: Compute the child and parent scores, then explain the limitation.


## Exercise 3 — Compare models

Fit `DummyRegressor`, `RandomForestRegressor`, and `XGBRegressor` on the shared training partition. Use validation data only for XGBoost early stopping. Report RMSE, MAE, and R² once on the untouched test partition. Keep all random seeds fixed.

In [ ]:
# TODO: Fit the three models and create a test-only metric table.


## Exercise 4 — Diagnose and describe scope

Plot actual against predicted rentals for XGBoost. Check residuals by hour or season. Write two sentences: one about the pattern you observe and one explaining why this experiment does not establish future-date forecasting performance.

In [ ]:
# TODO: Add a plot and a brief interpretation.
